# Golden Record Sample Exploration

This notebook is used to explore and evaluate sample educational-resource records using the Rubric Agent Level 1 and Level 2 record architecture.

The goal is to test how well the proposed structure handles variation across sources before finalizing schemas, controlled vocabularies, and validation rules.

## 1. Load Sample Records

Load the sample JSON records used for architecture testing and inspect their overall structure.

## 2. Inspect Level 1 Fields

Review the descriptive and identity metadata for each educational resource.

Level 1 answers:

> **What is this resource?**

Examples include identity, source, title, subject, education level, resource type, description, license, canonical URL, provenance, publisher, source identifiers, collection timestamps, and version information.

## 3. Inspect Level 2 Relationships

Review how each resource relates to other educational objects.

Level 2 answers:

> **How does this resource relate to other educational objects?**

Examples include:

- lesson belongs to unit
- lesson belongs to sequence
- worksheet belongs to lesson
- slide deck belongs to lesson
- quiz assesses lesson
- lesson addresses curriculum objective
- lesson precedes another lesson
- teacher guide supports lesson
- answer key answers worksheet

## 4. Compare Record Shapes

Compare the sample records to identify structural similarities and differences across sources and resource types.

Look for:

- fields shared across most records
- source-specific fields
- optional versus consistently available fields
- different representations of similar concepts
- different relationship patterns

## 5. Identify Missing or Ambiguous Fields

Identify metadata or relationships that are missing, unclear, difficult to normalize, or represented inconsistently.

Examples:

- unknown education level
- unknown license
- ambiguous parent resource
- missing canonical URL
- unresolved relationship target
- source terminology with no clear normalized equivalent

## 6. Validate Controlled Vocabulary Candidates

Identify fields that would benefit from machine-readable controlled vocabularies.

Candidate categories include:

- resource types
- material types
- education level types
- learning expectation types
- relationship types
- learner support types
- assessment types
- standards/framework types

Preserve source-native terminology while evaluating possible normalized values.

## 7. Record Architecture Observations

Document findings that may affect the Rubric Agent record architecture.

Track observations such as:

- fields that should be required or optional
- fields that belong specifically to Level 1 or Level 2
- useful normalization patterns
- relationship modeling issues
- source-specific exceptions
- fields that should remain free text
- controlled vocabulary candidates
- changes that may be needed before defining the formal schema

In [11]:
import os
from pathlib import Path

from dotenv import load_dotenv

from collectors.shared.storage import get_r2_client

PROJECT_ROOT = Path.cwd().parent
load_dotenv(PROJECT_ROOT / ".env")

R2_BUCKET = os.getenv("R2_BUCKET_NAME")
OAK_API_KEY = os.getenv("OAK_API_KEY")

print("R2 bucket:", R2_BUCKET)
print("Oak API key configured:", bool(OAK_API_KEY))


r2 = get_r2_client()

R2 bucket: rubric-agent-corpus
Oak API key configured: True


In [13]:
# check R2 creds

r2_response = r2.list_objects_v2(
    Bucket=R2_BUCKET,
    MaxKeys=1,
)

print("R2 status:", r2_response["ResponseMetadata"]["HTTPStatusCode"])

R2 status: 200


In [14]:
# check Oak
TEST_LESSON_SLUG = "count-tenths-in-different-ways"

oak_url = (
    "https://open-api.thenational.academy/api/v0/"
    f"lessons/{TEST_LESSON_SLUG}/assets"
)

oak_response = requests.get(
    oak_url,
    headers={"Authorization": f"Bearer {OAK_API_KEY}"},
    timeout=30,
)

print("Oak status:", oak_response.status_code)


Oak status: 200


## Notebook Checkpoint: Source Access Confirmed

The notebook environment is ready for golden-record sample exploration.

### Verified setup

- Project root is available from the notebook.
- Environment variables load successfully from `.env`.
- Cloudflare R2 client initializes successfully.
- R2 bucket `rubric-agent-corpus` is reachable.
- Oak API credentials are configured.
- Oak lesson asset endpoint is reachable.

Connectivity checks returned:

```text
R2: 200
Oak API: 200

In [16]:
# Oak bulk metadata currently stored in R2
import json

OAK_BULK_KEY = (
    "metadata/oak/math/primary/"
    "resource_19d7cfd594eb4ef3.json"
)

response = r2.get_object(
    Bucket=R2_BUCKET,
    Key=OAK_BULK_KEY,
)

oak_bulk = json.load(response["Body"])

print(type(oak_bulk))
print(oak_bulk.keys())

<class 'dict'>
dict_keys(['sequenceSlug', 'subjectTitle', 'sequence', 'lessons'])


In [18]:
SAMPLE_SIZE = 6

oak_lessons = oak_bulk["lessons"]

step = max(1, len(oak_lessons) // SAMPLE_SIZE)

sample_lessons = [
    oak_lessons[i]
    for i in range(0, len(oak_lessons), step)
][:SAMPLE_SIZE]

print("Sample size:", len(sample_lessons))

Sample size: 6


In [19]:
KNOWN_LESSON_SLUG = "count-tenths-in-different-ways"

known_lesson = next(
    lesson
    for lesson in oak_lessons
    if lesson.get("lessonSlug") == KNOWN_LESSON_SLUG
)

sample_lessons[0] = known_lesson

In [24]:
import pandas as pd

sample_rows = []

for lesson in sample_lessons:
    sample_rows.append({
        "lessonSlug": lesson.get("lessonSlug"),
        "lessonTitle": lesson.get("lessonTitle"),
        "subjectTitle": lesson.get("subjectTitle"),
        "keyStageTitle": lesson.get("keyStageTitle"),
        "unitTitle": lesson.get("unitTitle"),
        "downloadsAvailable": lesson.get("downloadsavailable"),
        "restricted": lesson.get("restricted"),
        "keywordCount": len(lesson.get("lessonKeywords") or []),
        "learningPointCount": len(lesson.get("keyLearningPoints") or []),
        "hasMisconceptions": bool(
            lesson.get("misconceptionsAndCommonMistakes")
        ),
        "hasTranscript": bool(
            lesson.get("transcript_sentences")
            or lesson.get("transcript_vtt")
        ),
    })

sample_df = pd.DataFrame(sample_rows)

display(sample_df)

,lessonSlug,lessonTitle,subjectTitle,keyStageTitle,unitTitle,downloadsAvailable,restricted,keywordCount,learningPointCount,hasMisconceptions,hasTranscript
0,count-tenths-in-different-ways,Count tenths in different ways,Maths,Key Stage 2,"Understand tenths as part of a whole, represen...",True,None,2,3,True,True
1,use-known-addition-and-subtraction-facts-withi...,Use known addition and subtraction facts withi...,Maths,Key Stage 1,Secure fluency of addition and subtraction fac...,True,None,1,3,True,True
2,use-addition-and-subtraction-to-solve-problems...,Use addition and subtraction to solve problems...,Maths,Key Stage 2,Informal and mental strategies for adding and ...,True,None,3,3,True,True
3,use-divisibility-rules-for-2-3-4-5-6-8-and-10-...,"Use divisibility rules for 2, 3, 4, 5, 6, 8 an...",Maths,Key Stage 2,"7 times table: odd and even patterns, square n...",True,None,2,4,True,True
4,solve-problems-involving-missing-coordinates,Solve problems involving missing coordinates,Maths,Key Stage 2,"Area, perimeter, position and direction",True,None,2,4,True,True
5,length-can-be-measured-in-metres-and-centimetres,Length can be measured in metres and centimetres,Maths,Key Stage 1,"Sense of measure - capacity, volume and mass",True,None,2,3,True,True


In [27]:
import pprint

def view_lesson(lesson):
    print(f"Title: {lesson.get('lessonTitle')}")
    print(f"Slug:  {lesson.get('lessonSlug')}")
    print()

    pprint.pprint(lesson)

view_lesson(sample_lessons[0])

Title: Count tenths in different ways
Slug:  count-tenths-in-different-ways

{'canonicalUrl': 'https://www.thenational.academy/teachers/programmes/maths-primary-ks2/units/understand-tenths-as-part-of-a-whole-represent-and-calculate-mentally/lessons/count-tenths-in-different-ways',
 'contentGuidance': None,
 'downloadsavailable': True,
 'keyLearningPoints': [{'keyLearningPoint': 'Tenths can be counted and '
                                            'represented with place value '
                                            'resources.'},
                       {'keyLearningPoint': 'Tenths can be counted and '
                                            'represented as decimals on a '
                                            'number line.'},
                       {'keyLearningPoint': 'Tenths can be counted and '
                                            'represented as fractions on a '
                                            'number line.'}],
 'keyStageSlug': 'ks2',
 'keyStag

In [39]:
from collections import Counter
import pandas as pd

keyword_counts = Counter(
    keyword.get("keyword")
    for lesson in oak_lessons
    for keyword in (lesson.get("lessonKeywords") or [])
    if keyword.get("keyword")
)

keyword_df = pd.DataFrame(
    keyword_counts.items(),
    columns=["keyword", "lesson_count"]
).sort_values(
    ["keyword"],
    key=lambda col: col.str.lower()
).reset_index(drop=True)

display(keyword_df)

,keyword,lesson_count
0,10 less,1
1,10 more,1
2,10 p,2
3,10 times table,1
4,100s boundary,6
...,...,...
782,y-axis,1
783,Y-axis,2
784,Year,2
785,Yesterday,1


In [40]:
KEYWORD_EXPORT = PROJECT_ROOT / "samples" / "oak_keyword_inventory.csv"

keyword_df.to_csv(
    KEYWORD_EXPORT,
    index=False
)

print(KEYWORD_EXPORT)

/home/nunto/dev/rubric-agent/samples/oak_keyword_inventory.csv


In [36]:
# Topic groups for sample lesson candidates

TOPIC_GROUPS = {
    "volume_or_cylinder": ["volume", "cylinder"],
    "zero": ["zero"],
    "unequal_or_congruent": ["unequal", "congruent"],
    "triangle": ["triangle"],
    "ratio": ["ratio"],
    "polygon_or_coefficient": ["polygon", "coefficient"],
    "equivalent_fractions": ["equivalent fraction"],
    "quadrant": ["quadrant"],
}

def get_lesson_keywords(lesson):
    return [
        keyword.get("keyword", "")
        for keyword in (lesson.get("lessonKeywords") or [])
    ]


def find_lessons_by_keywords(lessons, search_terms):
    matches = []

    for lesson in lessons:
        keywords = get_lesson_keywords(lesson)
        normalized_keywords = [keyword.lower() for keyword in keywords]

        matched_terms = [
            term
            for term in search_terms
            if any(
                term.lower() in keyword
                for keyword in normalized_keywords
            )
        ]

        if matched_terms:
            matches.append({
                "lessonSlug": lesson.get("lessonSlug"),
                "lessonTitle": lesson.get("lessonTitle"),
                "keyStageTitle": lesson.get("keyStageTitle"),
                "unitTitle": lesson.get("unitTitle"),
                "keywords": ", ".join(keywords),
                "matchedTerms": ", ".join(matched_terms),
            })

    return matches

topic_candidates = []

for topic_group, search_terms in TOPIC_GROUPS.items():
    matches = find_lessons_by_keywords(
        oak_lessons,
        search_terms,
    )

    for match in matches:
        match["topicGroup"] = topic_group
        topic_candidates.append(match)

topic_candidates_df = pd.DataFrame(topic_candidates)

display(
    topic_candidates_df[
        [
            "topicGroup",
            "lessonTitle",
            "keyStageTitle",
            "unitTitle",
            "matchedTerms",
            "keywords",
            "lessonSlug",
        ]
    ]
)


,topicGroup,lessonTitle,keyStageTitle,unitTitle,matchedTerms,keywords,lessonSlug
0,volume_or_cylinder,Solve problems involving volume,Key Stage 2,Measures: mass and capacity,volume,"Bar model, Whole, Part, Volume",solve-problems-involving-volume
1,volume_or_cylinder,Measuring the volume of liquids using millilit...,Key Stage 2,Measures: mass and capacity,volume,"Volume, Millilitre",measuring-the-volume-of-liquids-using-millilitres
2,volume_or_cylinder,Measure volume in whole litres and millilitres,Key Stage 2,Measures: mass and capacity,volume,"Volume, Millilitre, Litre",measure-volume-in-whole-litres-and-millilitres
3,volume_or_cylinder,Comparing and estimating mass and volume,Key Stage 2,Measures: mass and capacity,volume,"Estimate, Mass, Volume",comparing-and-estimating-mass-and-volume
4,volume_or_cylinder,Understanding capacity and volume,Key Stage 2,Measures: mass and capacity,volume,"Capacity, Volume, Milliliitre",understanding-capacity-and-volume
...,...,...,...,...,...,...,...
74,equivalent_fractions,Explain the relationship within families of eq...,Key Stage 2,Comparing fractions using equivalence and deci...,equivalent fraction,"Numerator, Denominator, Equivalent fraction, S...",explain-the-relationship-within-families-of-eq...
75,equivalent_fractions,Use understanding of equivalent fractions to s...,Key Stage 2,Comparing fractions using equivalence and deci...,equivalent fraction,"Equivalent fraction, Simplify, Scale up/down",use-understanding-of-equivalent-fractions-to-s...
76,equivalent_fractions,Explain the relationship between numerators an...,Key Stage 2,Comparing fractions using equivalence and deci...,equivalent fraction,"Numerator, Denominator, Equivalent fraction, S...",explain-the-relationship-between-numerators-an...
77,quadrant,Explain how negative numbers are used on a coo...,Key Stage 2,Negative numbers,quadrant,"Coordinates, Quadrant, Axis / Axes",explain-how-negative-numbers-are-used-on-a-coo...


In [38]:
KEYWORD_EXPORT = PROJECT_ROOT / "samples" / "oak_topic_inventory.csv"

topic_candidates_df.to_csv(
    KEYWORD_EXPORT,
    index=False
)

print(KEYWORD_EXPORT)

/home/nunto/dev/rubric-agent/samples/oak_topic_inventory.csv


In [44]:
# pick sampling
SAMPLE_LESSON_SLUGS = [
    "explore-recognise-and-compare-three-different-3d-shapes",
    "explain-the-size-of-a-part-in-relation-to-the-whole",
    "solve-problems-involving-missing-coordinates",
    "compose-tangram-images",
    "combine-multiplication-with-addition-and-subtraction",
    "draw-polygons-specified-by-coordinates-in-the-first-quadrant",
]

lesson_lookup = {
    lesson.get("lessonSlug"): lesson
    for lesson in oak_lessons
}

sample_lessons = [
    lesson_lookup[slug]
    for slug in SAMPLE_LESSON_SLUGS
]

print("Selected lessons:", len(sample_lessons))

Selected lessons: 6


In [46]:
# level 1: resource identity
def build_level_1_record(lesson):
    return {
        "record_id": f"ra_lesson_oak_{lesson.get('lessonSlug')}",
        "level": 1,
        "record_type": "lesson",

        "source": {
            "name": "oak",
            "source_id": lesson.get("lessonSlug"),
            "source_url": lesson.get("oakUrl"),
            "canonical_url": lesson.get("canonicalUrl"),
        },

        "identity": {
            "title": lesson.get("lessonTitle"),
            "subject": lesson.get("subjectTitle"),
            "key_stage": lesson.get("keyStageTitle"),
            "unit_title": lesson.get("unitTitle"),
        },

        "learning": {
            "keywords": lesson.get("lessonKeywords") or [],
            "key_learning_points": lesson.get("keyLearningPoints") or [],
            "pupil_lesson_outcome": lesson.get("pupilLessonOutcome"),
            "misconceptions": lesson.get(
                "misconceptionsAndCommonMistakes"
            ),
        },

        "availability": {
            "downloads_available": lesson.get("downloadsavailable"),
            "restricted": lesson.get("restricted"),
        },
    }

level_1_records = [
    build_level_1_record(lesson)
    for lesson in sample_lessons
]

print("Level 1 records:", len(level_1_records))
pprint.pprint(level_1_records[0])

Level 1 records: 6
{'availability': {'downloads_available': True, 'restricted': None},
 'identity': {'key_stage': 'Key Stage 1',
              'subject': 'Maths',
              'title': 'Explore, recognise and compare three different 3D '
                       'shapes',
              'unit_title': 'Recognise, compose, decompose and manipulate 2D '
                            'and 3D shapes'},
 'learning': {'key_learning_points': [{'keyLearningPoint': 'Recognise objects '
                                                           'that are similar '
                                                           'shapes.'},
                                      {'keyLearningPoint': '3D shapes have 3 '
                                                           'dimensions and are '
                                                           'not flat.'},
                                      {'keyLearningPoint': 'A sphere is shaped '
                                                           